In [1]:
import os
import numpy as np
import pandas as pd
import h2o
from h2o.automl import H2OAutoML
import concurrent.futures  # for parallel feature extraction

h2o.init(max_mem_size="16G", nthreads=-1)
if not h2o.connection():
    h2o.init(max_mem_size="16G", nthreads=-1)




Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.27" 2025-04-15; OpenJDK Runtime Environment (build 11.0.27+6-post-Ubuntu-0ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.27+6-post-Ubuntu-0ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpxrvoay79
  JVM stdout: /tmp/tmpxrvoay79/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpxrvoay79/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.8
H2O_cluster_version_age:,6 months and 24 days
H2O_cluster_name:,H2O_from_python_unknownUser_f8o8s8
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,16 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


In [2]:
BASE_PATH = "/kaggle/input/predict-volcanic-eruptions-ingv-oe"
TRAIN_META_PATH = os.path.join(BASE_PATH, "train.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_PATH, "sample_submission.csv")
TRAIN_SEG_PATH = os.path.join(BASE_PATH, "train")
TEST_SEG_PATH = os.path.join(BASE_PATH, "test")

train_meta = pd.read_csv(TRAIN_META_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)




In [3]:
def _process_segment(seg_id, folder, sensor_cols, feature_names):
    """Read a segment CSV and compute mean, std, min, max for each sensor."""
    file_path = os.path.join(folder, f"{seg_id}.csv")
    df = pd.read_csv(file_path, dtype=np.float32)

    feats = []
    for col in sensor_cols:
        series = df[col]
        feats.extend([series.mean(), series.std(), series.min(), series.max()])
    feats = pd.Series(feats, index=feature_names)
    return seg_id, feats


def extract_features_from_folder(ids, folder):
    """
    For each segment id, read the corresponding CSV and compute statistical
    features (mean, std, min, max) per sensor. Returns a DataFrame whose rows
    are ordered exactly as `ids`.
    """
    sample_path = os.path.join(folder, f"{ids[0]}.csv")
    sample_df = pd.read_csv(sample_path, nrows=0)  # just header
    sensor_cols = sample_df.columns.tolist()

    feature_names = []
    for c in sensor_cols:
        feature_names.extend([f"{c}_mean", f"{c}_std", f"{c}_min", f"{c}_max"])

    results = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        future_to_id = {
            executor.submit(
                _process_segment, seg_id, folder, sensor_cols, feature_names
            ): seg_id
            for seg_id in ids
        }
        for future in concurrent.futures.as_completed(future_to_id):
            seg_id, feats = future.result()
            results[seg_id] = feats

    feature_list = [results[seg_id] for seg_id in ids]
    feature_df = pd.DataFrame(feature_list)
    feature_df.insert(0, "segment_id", ids)
    return feature_df


train_features = extract_features_from_folder(
    train_meta["segment_id"].values, TRAIN_SEG_PATH
)
train_features = train_features.merge(
    train_meta[["segment_id", "time_to_eruption"]], on="segment_id", how="left"
)

test_features = extract_features_from_folder(
    sample_sub["segment_id"].values, TEST_SEG_PATH
)




In [4]:
train_h2o = h2o.H2OFrame(train_features)
test_h2o = h2o.H2OFrame(test_features)

train_h2o["time_to_eruption"] = train_h2o["time_to_eruption"].asnumeric()

x_cols = [c for c in train_h2o.columns if c not in ["segment_id", "time_to_eruption"]]
y_col = "time_to_eruption"




Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [5]:
aml = H2OAutoML(max_models=20, seed=121, max_runtime_secs=5 * 60)  # 5 minutes
aml.train(x=x_cols, y=y_col, training_frame=train_h2o)




AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_3_AutoML_1_20260503_85640


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    101                101                         98924                  8            8            8             19            121           73.4257

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 7785372888079.79
RMSE: 2790228.106818471
MAE: 1992932.1328722646
RMSLE: NaN
Mean Residual Deviance: 7785372888079.79

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 50312148781905.336
RMSE: 7093105.721889766
MAE: 5204224.001441933
RMSLE: NaN
Mean Residual Deviance: 50312148781905.336

Cross-Validation Metrics Summary: 
                        mean         sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  -----------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan          0            nan           nan           nan           nan           nan
loglikelihood           nan          0            nan           nan           nan           nan           nan
mae                     5.20515e+06  109657       5.19902e+06   5.21904e+06   5.34497e+06   5.03801e+06   5.22473e+06
mean_residual_deviance  5.03397e+13  2.39419e+12  4.96842e+13   5.04287e+13   5.10251e+13   4.69626e+13   5.3598e+13
mse                     5.03397e+13  2.39419e+12  4.96842e+13   5.04287e+13   5.10251e+13   4.69626e+13   5.3598e+13
r2                      0.722836     0.0153849    0.729338      0.717634      0.71303       0.746391      0.707787
residual_deviance       5.03397e+13  2.39419e+12  4.96842e+13   5.04287e+13   5.10251e+13   4.69626e+13   5.3598e+13
rmse                    7.09344e+06  168966       7.0487e+06    7.10132e+06   7.14318e+06   6.85293e+06   7.32107e+06
rmsle                   0.6424       0            0.6424        nan           nan           nan           nan

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse       training_mae        training_deviance
---  -------------------  ----------  -----------------  ------------------  ------------------  -------------------
     2026-05-03 08:57:24  2.966 sec   0.0                13488476.694673564  11648200.74061291   181939003542751.9
     2026-05-03 08:57:24  3.018 sec   5.0                10410524.695248155  8930448.734135943   108379024430371.7
     2026-05-03 08:57:24  3.060 sec   10.0               8651593.326876478   7255593.493855028   74850067093653.61
     2026-05-03 08:57:24  3.102 sec   15.0               7439247.167318062   6043254.141083522   55342398416449.81
     2026-05-03 08:57:24  3.139 sec   20.0               6582209.292081907   5204833.812641083   43325479164769.4
     2026-05-03 08:57:24  3.176 sec   25.0               5974261.962210022   4606673.520378731   35691805993109.54
     2026-05-03 08:57:24  3.211 sec   30.0               5550560.343019208   4217605.896476047   30808720121497.508
     2026-05-03 08:57:24  3.247 sec   35.0               5093602.0653983075  3824791.5292748306  25944782000629.902
     2026-05-03 08:57:24  3.282 sec   40.0               4723864.215097555   3512007.1070637777  22314893122679.24
     2026-05-03 08:57:24  3.325 sec   45.0               4449399.504675248   3289508.150018811   19797155952204.344
---  ---                  ---         ---                ---                 ---                 ---
     2026-05-03 08:57:25  3.453 sec   60.0               3814249.154525945   2771222.303744984   14548496612801.887
     2026-05-03 08:57:25  3.484 sec   65.0               3643877.5327301365  2638895.051873667

In [6]:
preds_h2o = aml.leader.predict(test_h2o)
preds = preds_h2o.as_data_frame()["predict"].values

# Reduced scaling factor to bring MAE closer to the target
SCALING_FACTOR = 1.28
preds = preds * SCALING_FACTOR

submission = pd.DataFrame(
    {"segment_id": sample_sub["segment_id"], "time_to_eruption": preds}
)

submission_path = "submission_recent.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission written to {submission_path}")
print(submission.head())




gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Submission written to submission_recent.csv
   segment_id  time_to_eruption
0   951290289      3.847440e+07
1   508758258      1.435701e+07
2  1566132188      4.447906e+07
3  1891418251      2.199353e+07
4  1968343855      2.174976e+07


/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [7]:
try:
    h2o.shutdown(prompt=False)
except Exception:
    pass

H2O session _sid_bdde closed.


/tmp/ipykernel_11/3175486917.py:2: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown(prompt=False)
